# 第四章 深度学习框架 PyTorch

我们知道，深度神经网络基本可以通过一些数组操作来表示。而 NumPy 提供了高效的数组操作接口。我们确实可以使用 NumPy 编写出高性能的神经网络推理与训练程序，但是这依然十分复杂。

- 我们需要手动推导每个参数数组的梯度公式。也就是说，我们的模型只要变化了，我们就需要重新推导，并且重新实现。
- 我们可以利用链式法则，只实现每层的梯度计算，随后递归调用。但如果你想实现一个新层，除非这个新层实际上只是已有层的复合层，否则仍然需要手动实现梯度计算。

为了使得开发更加简单和灵活，深度学习框架被构建。

## 在本章开始之前

本章开始，你大抵是需要一台配备有 Nvidia GPU 的机器。没有英伟达 GPU 设备，肯定会影响你学习本章内容，因为其中的一些代码你将无法实践。你可以让全部运算跑在 CPU 上，这可能会很慢。并且，如果你选择跑在 CPU 上，那么你最好拥有 ≥16GB 的 RAM。

## 深度学习框架

深度学习框架可以快速地构建深度学习模型（其实不止是深度学习模型）。你只需要实现前向传播代码，计算机就可以自动推导它的梯度计算方法。因此你不需要考虑如何求导你的模型。

深度学习框架还实现了大量常用的模型和组件，例如 MSE 损失函数、交叉熵损失函数、卷积神经网络、Adam 优化器等。

现代深度学习框架甚至还实现了分布式相关的通信操作，方便大规模推理或者训练。

深度学习框架对于不同硬件平台做了深层次的优化，使得其在深度学习场景下，往往比 NumPy 更快。并且，其能方便调用 Nvidia GPU 等加速硬件。

## 计算图自动微分技术

### 自动微分

深度学习框架的核心是**自动微分**。

我们有一个表达式 $f(x)$，它的参数只有 $x$。我们想要求 $x$ 的微分，朴素的思路是使用定义法：

$$f'(x_0) \approx \frac{f(x_0 + \delta x) - f(x_0)}{\delta x}$$

其中 $\delta x$ 要足够小。使用代码实现：

In [8]:
from typing import Callable

def differ(f: Callable[[float], float], x: float, epsilon: float = 1e-6) -> float:
    return (f(x + epsilon) - f(x)) / epsilon

def f(x: float) -> float:
    return x ** 2 + 2 * x

print(differ(f, 1.))  # ≈ 4

4.000000999759834


通过这种方法，我们可以让计算机估算出每个变量的微分。让计算机可以计算函数的微分，这称之为**自动微分**。

但是这种方法有一定的缺陷。当参数量很大时，逐个计算是很慢的。并且对于很深的嵌套函数，会有大量重复的计算。

除了定义法之外，我们还可以使用**导函数法**。我们可以提前计算出函数的导函数。例如，$f(x) = x^2 + 2x$ 的导函数为 $f'(x) = 2x + 2$。

更好的是，一些函数的导函数十分简单。例如 sigmoid 函数与 ReLU 函数。因此，现代自动微分技术采用**导函数法**。

### 计算图

计算图是对算式的一种建模方法，便于计算机处理。计算图中有两种节点，分别是**数据**和**操作**。边代表输入输出，或者说是依赖关系。

算式 $f(x, y, z) = x^2 + y^2 + x^2y^3 + z$ 的计算图为：

（图片见原文档）

对于计算机来说，我们使用面向对象的方式，只需要实现 Node 类，随后继承出 Variable 类和 Operation 类即可表示计算图。

我们可以使用运算符重载的方法来捕捉算式构建时的依赖关系，得知计算图的拓扑结构。

### 计算图与链式法则自动微分

对于复杂的算式求导是很难的，但是对于单个操作求导是很简单的。例如乘法求导，结果就是系数；加法求导，结果是 1。

我们可以让计算机计算出每个 $\frac{\partial op_i}{\partial op_{i-1}}$，然后将每一项相乘，即可得到我们需要的结果。

在实际实现的过程中，算法会对计算图进行**反向传播**，按照一定顺序遍历计算图，不断累乘梯度值，当遍历完成时，也就求出了每个参数的导数值。

## PyTorch 语法入门

PyTorch 是目前最常用的深度学习框架。torch 中的基本类型是 `torch.Tensor`，与 NumPy 中的 `numpy.ndarray` 对标。相关语法与 `ndarray` 几乎相同。

### Tensor 张量

PyTorch 中的张量使用方式与 NumPy 中的 ndarray 大量雷同。torch 中的张量使用 `torch.tensor` 定义：

In [9]:
import torch

tsr = torch.tensor([1, 2, 3], dtype=torch.float16)
print(tsr)

tensor([1., 2., 3.], dtype=torch.float16)


Tensor 可以通过 `device` 参数指定 tensor 所在的设备。

> **设备**：对于很多异构计算结构的机器来说，它不止拥有 CPU，还有 GPU、NPU、TPU 等异构加速设备。你需要指定张量是在哪个设备上定义的。

In [10]:
import torch

# 在 CPU 上创建张量
tsr_cpu = torch.tensor([1, 2, 3], dtype=torch.float16)
print(f"CPU tensor: {tsr_cpu}")

# 注意：以下代码需要在有 CUDA GPU 的设备上运行
# tsr_cuda = torch.tensor([1, 2, 3], dtype=torch.float16, device='cuda')
# print(f"CUDA tensor: {tsr_cuda}")

CPU tensor: tensor([1., 2., 3.], dtype=torch.float16)


### nn.Module 模块

`torch.nn.Module` 是 PyTorch 的核心组件。它是对模型中模块的抽象建模（模型本身也可以看作一个 Module）。类似于函数，`nn.Module` 有输入与输出，并且拥有参数。

继承 `nn.Module` 后，你需要实现 `forward` 方法。例如：

In [12]:
import torch
import torch.nn as nn

class Net(nn.Module):
    def __init__(self, n: int) -> None:
        super().__init__()  # 先执行父类的init
        self.n = n
        
        # 你可以使用nn.Parameter来包装模型的参数
        self.weights = nn.Parameter(torch.randn(n))
        
        # 使用register_buffer方法来注册不需要更新的张量
        self.register_buffer('no_grad_tensor', torch.randn(n))
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * self.weights + self.no_grad_tensor

net = Net(128)
x = torch.randn(128)
y = net(x)
print(f"Output shape: {y.shape}")

Output shape: torch.Size([128])


> **推荐使用 `nn.Parameter` 和 `register_buffer` 来定义数据**，防止在 Module 嵌套关系复杂之后，torch 参数推导出现问题。

你可以使用 `parameters` 方法来获取 `nn.Module` 中所有注册的参数：

In [13]:
import torch
import torch.nn as nn

class Model(nn.Module):
    def __init__(self, n: int) -> None:
        super().__init__()
        self.w1 = nn.Parameter(torch.randn(n))
        self.w2 = nn.Parameter(torch.randn(n))
        self.b = nn.Parameter(torch.randn(n))
    
    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        return x1 * self.w1 + x2 * self.w2 + self.b

model = Model(128)
print("Parameters:")
for p in model.parameters():
    print(f"  Shape: {p.shape}")

print("\nNamed Parameters:")
for name, p in model.named_parameters():
    print(f"  {name}: {p.shape}")

Parameters:
  Shape: torch.Size([128])
  Shape: torch.Size([128])
  Shape: torch.Size([128])

Named Parameters:
  w1: torch.Size([128])
  w2: torch.Size([128])
  b: torch.Size([128])


模型可以整体转移到某个 device 上，使用 `to` 方法：

In [14]:
import torch
import torch.nn as nn

class Model(nn.Module):
    def __init__(self, n: int) -> None:
        super().__init__()
        self.w1 = nn.Parameter(torch.randn(n))
        self.w2 = nn.Parameter(torch.randn(n))
        self.b = nn.Parameter(torch.randn(n))
    
    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        return x1 * self.w1 + x2 * self.w2 + self.b

# 注意：以下代码需要在有 CUDA GPU 的设备上运行
# model = Model(128)
# model.to('cuda')  # 转移到cuda加速设备上
# print(next(model.parameters()).device)  # cuda:0

### 损失函数

PyTorch 中定义损失函数比较灵活。比较推荐的方式是使用 `nn` 中自带的一些损失函数：

In [15]:
import torch
import torch.nn as nn

# 交叉熵损失
criterion = nn.CrossEntropyLoss()
logits = torch.randn(4, 128)  # shape: (batch_size, features)
targets = torch.tensor([0, 99, 67, 2])  # batch中每个正确类别的索引

loss = criterion(logits, targets)
print(f"CrossEntropyLoss: {loss.item():.4f}")

# MSE 均方损失
criterion_mse = nn.MSELoss()
logits = torch.randn(4, 128)
targets = torch.randn(4, 128)  # 均方损失函数的targets输入不是索引

loss_mse = criterion_mse(logits, targets)
print(f"MSELoss: {loss_mse.item():.4f}")

CrossEntropyLoss: 5.9504
MSELoss: 1.9176


其实，你不用 `nn` 自带的也是可以的。只要是运算就行：

In [16]:
import torch
import torch.nn as nn

class CustomLoss(nn.Module):  # 使用nn.Module
    def __init__(self) -> None:
        super().__init__()
    
    def forward(self, predictions: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        return torch.mean((predictions - targets) ** 2)

def custom_loss(predictions: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    return torch.mean((predictions - targets) ** 2)

predictions = torch.randn(32, 128)
targets = torch.randn(32, 128)

criterion = CustomLoss()
loss1 = criterion(predictions, targets)
print(f"CustomLoss (nn.Module): {loss1.item():.4f}")

loss2 = custom_loss(predictions, targets)
print(f"CustomLoss (function): {loss2.item():.4f}")

loss3 = torch.mean((predictions - targets) ** 2)
print(f"Direct computation: {loss3.item():.4f}")

CustomLoss (nn.Module): 1.9977
CustomLoss (function): 1.9977
Direct computation: 1.9977


### 自动求导

torch 中根据某个值反向计算出梯度，使用 `backward` 方法：

In [17]:
import torch

W = torch.randn(3, 4, requires_grad=True)
b = torch.randn(4, requires_grad=True)
x = torch.randn(2, 3)
y = torch.randn(2, 4)

pred = x @ W + b

loss = torch.mean((pred - y) ** 2)
loss.backward()

print(f"W.grad shape: {W.grad.shape}")
print(f"b.grad shape: {b.grad.shape}")

W.grad shape: torch.Size([3, 4])
b.grad shape: torch.Size([4])


> **关于 `requires_grad` 参数**：该参数用于指定张量在反向传播时是否需要计算梯度。被 `nn.Parameter` 包装的张量，它的 `requires_grad` 自动为 True。

In [18]:
import torch
import torch.nn as nn

# 普通张量 requires_grad 默认为 False
w_no_grad = torch.randn(2, 3)
w_with_grad = torch.randn(2, 3, requires_grad=True)

print(f"w_no_grad.requires_grad: {w_no_grad.requires_grad}")
print(f"w_with_grad.requires_grad: {w_with_grad.requires_grad}")

# nn.Parameter 自动设置 requires_grad 为 True
w_param = nn.Parameter(torch.randn(2, 3))
print(f"w_param.requires_grad: {w_param.requires_grad}")

w_no_grad.requires_grad: False
w_with_grad.requires_grad: True
w_param.requires_grad: True


### Optimizer 优化器

优化器用于优化一组参数。创建它需要输入被优化的参数。

使用 `step` 方法进行一步优化：

In [19]:
import torch
import torch.nn as nn
from torch.optim import SGD

# 定义一个简单的模型
model = nn.Linear(10, 2)
optimizer = SGD(model.parameters(), lr=0.01, momentum=0.9)

# 模拟训练步骤
x = torch.randn(32, 10)
y = torch.randint(0, 2, (32,))

# 前向传播
pred = model(x)
loss = nn.CrossEntropyLoss()(pred, y)

# 反向传播
loss.backward()

# 优化一步
optimizer.step()

print(f"Loss: {loss.item():.4f}")
print("参数已更新")

Loss: 0.6943
参数已更新


### 数据组织

PyTorch 提供了两个极其核心的工具类：`Dataset` 和 `DataLoader`。

#### 1. Dataset 数据集

`torch.utils.data.Dataset` 是一个抽象类，只要你继承了这个类，并实现了两个核心魔法方法，PyTorch 就承认这是一个合法的 Dataset：

1. `__len__`: 告诉系统这个数据集一共有多少个样本。
2. `__getitem__`: 告诉系统当给定一个索引 `idx` 时，应该返回什么样的数据和标签。

In [1]:
import torch
from torch.utils.data import Dataset

class MyCustomDataset(Dataset):
    def __init__(self, num_samples=1000):
        super().__init__()
        # 这里为了演示，我们直接生成一些模拟的特征和标签
        self.num_samples = num_samples
        self.features = torch.randn(num_samples, 10)  # 10维特征
        self.labels = torch.randint(0, 2, (num_samples,))  # 0或1的二分类标签
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        feature = self.features[idx]
        label = self.labels[idx]
        return feature, label

# 实例化数据集
my_dataset = MyCustomDataset(num_samples=100)
print(f"数据集大小: {len(my_dataset)}")

# 测试抽取第 0 个样本
first_feature, first_label = my_dataset[0]
print(f"第一个样本特征 shape: {first_feature.shape}, 标签: {first_label}")

数据集大小: 100
第一个样本特征 shape: torch.Size([10]), 标签: 0


#### 2. DataLoader 数据加载器

`torch.utils.data.DataLoader` 包装了 Dataset，在后台帮你处理所有的批次拼接、打乱以及多进程加载工作。

In [2]:
from torch.utils.data import DataLoader

# 将之前定义好的 dataset 喂给 DataLoader
# batch_size=16 意味着每次吐出 16 个样本
# shuffle=True 意味着在每个 epoch 开始时打乱数据顺序
train_loader = DataLoader(dataset=my_dataset, batch_size=16, shuffle=True, num_workers=0)

# DataLoader 是一个可迭代对象
for batch_idx, (batch_features, batch_labels) in enumerate(train_loader):
    print(f"Batch {batch_idx}:")
    print(f"  Features shape: {batch_features.shape}")  # 形状会变成 (16, 10)
    print(f"  Labels shape: {batch_labels.shape}")  # 形状会变成 (16,)
    if batch_idx >= 2:  # 只打印前3个batch
        break

Batch 0:
  Features shape: torch.Size([16, 10])
  Labels shape: torch.Size([16])
Batch 1:
  Features shape: torch.Size([16, 10])
  Labels shape: torch.Size([16])
Batch 2:
  Features shape: torch.Size([16, 10])
  Labels shape: torch.Size([16])


#### 3. 将 DataLoader 融入训练循环

In [3]:
import torch.nn as nn
from torch.optim import SGD

# 使用之前创建的数据集
train_loader = DataLoader(dataset=my_dataset, batch_size=32, shuffle=True, num_workers=0)

model = nn.Linear(10, 2)
loss_function = nn.CrossEntropyLoss()
optimizer = SGD(model.parameters(), lr=1e-3, momentum=0.9)

epochs = 2
for epoch in range(epochs):
    total_loss = 0.0
    for batch_features, batch_labels in train_loader:
        
        # 1. 前向传播
        pred = model(batch_features)
        
        # 2. 计算损失
        loss = loss_function(pred, batch_labels)
        total_loss += loss.item()
        
        # 3. 反向传播与优化
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1}/{epochs} - Average Loss: {avg_loss:.4f}")

Epoch 1/2 - Average Loss: 0.8336
Epoch 2/2 - Average Loss: 0.8746


## Quick Start

下面，我们使用一个例子来快速了解一个 PyTorch 深度学习应用的组成。

我们使用 PyTorch 实现一个手写数字识别任务（MNIST）。

### 定义模型

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MLP(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        
        # 手写数字图片大小是 28x28，展平后是 784 维的一维向量
        self.flatten = nn.Flatten()
        self.linear1 = nn.Linear(784, 256)
        self.linear2 = nn.Linear(256, 128)
        self.linear3 = nn.Linear(128, 10)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.flatten(x)
        x = self.linear1(x)
        x = F.relu(x)
        x = self.linear2(x)
        x = F.relu(x)
        x = self.linear3(x)
        return x

model = MLP()
print(model)

MLP(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear1): Linear(in_features=784, out_features=256, bias=True)
  (linear2): Linear(in_features=256, out_features=128, bias=True)
  (linear3): Linear(in_features=128, out_features=10, bias=True)
)


### 数据准备与组织

我们直接使用 `torchvision` 来下载 MNIST 数据集：

In [15]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.ToTensor()

# 首次运行时会自动下载数据集
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

使用 `DataLoader` 来包装数据集：

In [16]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

### 实例化模型、损失函数与优化器

In [17]:
model = MLP()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

### 加速设备识别与迁移

我们经常需要识别所运行的平台拥有什么样的加速硬件，然后再将模型等张量迁移到设备上来调用硬件加速计算能力。

In [18]:
def get_device():
    if torch.cuda.is_available():
        device = torch.device('cuda')
        print(f"已挂载 GPU: {torch.cuda.get_device_name(0)}")
    
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
        print("已挂载 Apple MPS 加速")
    
    else:
        device = torch.device('cpu')
        print("未检测到加速硬件，使用 CPU 进行计算")
    
    return device

device = get_device()
model.to(device) 

已挂载 GPU: Tesla T4


MLP(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear1): Linear(in_features=784, out_features=256, bias=True)
  (linear2): Linear(in_features=256, out_features=128, bias=True)
  (linear3): Linear(in_features=128, out_features=10, bias=True)
)

### 训练代码

In [19]:
from tqdm import tqdm
epochs = 5

for epoch in range(epochs):
    model.train()
    train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Train]')
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        
        train_pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        test_pbar = tqdm(test_loader, desc=f'Epoch {epoch+1}/{epochs} [Test ]')
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += criterion(output, target).item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    test_loss /= len(test_loader)
    accuracy = 100. * correct / len(test_loader.dataset)
    print(f'\n>>> Epoch {epoch+1} Summary: Test Loss: {test_loss:.4f}, Accuracy: {accuracy:.2f}%\n')


Epoch 2/5 [Train]:   0%|          | 0/938 [03:10<?, ?it/s, loss=0.0071]


































































































































































































































































































































































































































































































































































































































































































































































































































































































































































>>> Epoch 1 Summary: Test Loss: 0.1260, Accuracy: 96.14%



Epoch 1/5 [Train]:   0%|          | 0/938 [00:14<?, ?it/s, loss=0.0869]

Epoch 1/5 [Test ]:   0%|          | 0/40 [00:14<?, ?it/s]



>>> Epoch 2 Summary: Test Loss: 0.1079, Accuracy: 96.40%





Epoch 2/5 [Train]:   0%|          | 0/938 [00:14<?, ?it/s, loss=0.0147]

































































































































































































































































































































































































































































































































































































































































































































































































































































































































































>>> Epoch 3 Summary: Test Loss: 0.0809, Accuracy: 97.50%




Epoch 3/5 [Train]:   0%|          | 0/938 [00:13<?, ?it/s, loss=0.0481]


































































































































































































































































































































































































































































































































































































































































































































































































































































































































































>>> Epoch 4 Summary: Test Loss: 0.0988, Accuracy: 96.99%



Epoch 4/5 [Train]:   0%|          | 0/938 [00:13<?, ?it/s, loss=0.0044]

Epoch 4/5 [Test ]:   0%|          | 0/40 [00:13<?, ?it/s]



>>> Epoch 5 Summary: Test Loss: 0.0710, Accuracy: 97.93%



### 模型保存与加载

PyTorch 提供了多种模型保存和加载的方式：

In [ ]:
# 保存整个模型
# torch.save(model, 'model.pth')

# 加载模型
# model = torch.load('model.pth')

# 推荐的方式：只保存参数
# torch.save(model.state_dict(), 'model_weights.pth')
# model = MLP()
# model.load_state_dict(torch.load('model_weights.pth'))

print("模型保存加载代码见上方注释")

---

*本章内容由 131AIClub（东南大学人工智能协会）编写*
*Missing Semester for AI - 人间烟火，亦是星辰*